In [ ]:
# TemporalBench v1 — Baseline Tier
# Difficulty: Well-separated validity windows, no noise injection
# Task families: AsOfQA, ChangeDetection, CausalQuery

!pip install -q "protobuf==5.29.6" kaggle-benchmarks
import kaggle_benchmarks as kbench

In [ ]:
import json
import numpy as np
from typing import Dict, List, Optional
from dataclasses import dataclass

DATA_PATH = '/kaggle/input/datasets/zacharymaronek/attention-time/benchmarks/v1_seed0/'

def load_benchmark(path: str):
    with open(path + "questions.jsonl") as f:
        questions = [json.loads(line) for line in f]
    with open(path + "facts.jsonl") as f:
        facts_data = [json.loads(line) for line in f]
    with open(path + "events.jsonl") as f:
        events_data = [json.loads(line) for line in f]
    return questions, facts_data, events_data

questions, facts_data, events_data = load_benchmark(DATA_PATH)
print(f'Loaded {len(questions)} questions, {len(facts_data)} facts, {len(events_data)} events')

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional
import numpy as np

@dataclass
class Fact:
    key: str
    value: str
    valid_from: int
    valid_to: int
    accesses: int = 0

    def is_valid_at(self, day: Optional[int]) -> bool:
        if day is None:
            return False
        if not isinstance(day, (int, float)):
            return False
        return self.valid_from <= int(day) <= self.valid_to

    def age(self, day: Optional[int]) -> float:
        if day is None or not isinstance(day, (int, float)):
            return float("inf")
        return int(day) - self.valid_from

    def decay_score(self, day: Optional[int], half_life: float = 50.0) -> float:
        if day is None or not isinstance(day, (int, float)):
            return 0.0
        return float(np.exp(-0.693 * self.age(day) / half_life))

    def attention_score(self) -> float:
        return 1.0 + 0.001 * np.log1p(self.accesses)


class TemporalAttentionStore:
    def __init__(
        self,
        half_life: float = 50.0,
        temporal_weight: float = 0.9,
        attention_weight: float = 0.1,
    ):
        self.half_life = half_life
        self.temporal_weight = temporal_weight
        self.attention_weight = attention_weight
        self.facts: Dict[str, List[Fact]] = {}

    def put(
        self,
        key: str,
        value: str,
        valid_from: int,
        valid_to: int,
        accesses: int = 0,
    ):
        if key not in self.facts:
            self.facts[key] = []
        self.facts[key].append(
            Fact(key, value, int(valid_from), int(valid_to), int(accesses))
        )

    def get(self, key: str, as_of_day: Optional[int] = None) -> Optional[str]:
        if key not in self.facts:
            return None
        if as_of_day is None or not isinstance(as_of_day, (int, float)):
            return None

        as_of_day = int(as_of_day)
        candidates = [f for f in self.facts[key] if f.is_valid_at(as_of_day)]
        if not candidates:
            return None

        scored = [
            (
                self.temporal_weight * f.decay_score(as_of_day, self.half_life)
                + self.attention_weight * f.attention_score(),
                f,
            )
            for f in candidates
        ]
        return max(scored, key=lambda x: x[0])[1].value


class TemporalAttentionStoreWithValidity:
    def __init__(self, half_life: float = 50.0):
        self.half_life = half_life
        self.facts: Dict[str, List[Fact]] = {}

    def put(
        self,
        key: str,
        value: str,
        valid_from: int,
        valid_to: int,
        accesses: int = 0,
    ):
        if key not in self.facts:
            self.facts[key] = []
        self.facts[key].append(
            Fact(key, value, int(valid_from), int(valid_to), int(accesses))
        )

    def get(self, key: str, as_of_day: Optional[int] = None) -> Optional[str]:
        if key not in self.facts:
            return None
        if as_of_day is None or not isinstance(as_of_day, (int, float)):
            return None

        as_of_day = int(as_of_day)
        candidates = [f for f in self.facts[key] if f.is_valid_at(as_of_day)]
        if not candidates:
            return None

        scored = [
            (f.decay_score(as_of_day, self.half_life) * f.attention_score(), f)
            for f in candidates
        ]
        return max(scored, key=lambda x: x[0])[1].value

In [ ]:
import json
from typing import Dict, List, Optional
import numpy as np

def parse_fact_content(content: str):
    parts = content.split(":")
    if len(parts) < 4:
        raise ValueError(f"Unexpected fact content format: {content}")
    domain = parts[0]
    subject = parts[1]
    key = f"{domain}:{subject}"
    value = content
    return key, value


def extract_day_from_dict(d: Dict, default: Optional[int] = None) -> Optional[int]:
    """Try multiple field names for a day/time value."""
    for field in ["t_event", "day", "as_of_day", "t", "time", "timestamp"]:
        val = d.get(field)
        if isinstance(val, (int, float)):
            return int(val)
    return default


def build_store(facts_data, events_data, store_class):
    store = store_class()
    event_log = []
    for ev in events_data:
        event_log.append({
            "type": ev.get("event_type"),
            "subject": ev.get("subject"),
            "day": extract_day_from_dict(ev),
            "content": ev.get("content"),
            "domain": ev.get("domain"),
            "value": ev.get("value"),
            "t_event": extract_day_from_dict(ev),
        })
    for fact in facts_data:
        key, value = parse_fact_content(fact["content"])
        valid_from = int(fact["t_valid_from"])
        valid_to = int(fact["t_valid_until"]) if fact.get("t_valid_until") is not None else 2**31 - 1
        accesses = int(fact.get("accesses", 0))
        store.put(key, value, valid_from, valid_to, accesses)
    return store, event_log


def replay_events(store, event_log: List[Dict], up_to_day: int):
    up_to_day = int(up_to_day)
    for ev in event_log:
        t_event = ev.get("t_event")
        if not isinstance(t_event, (int, float)):
            continue
        t_event = int(t_event)
        if t_event > up_to_day:
            break
        if ev.get("type") == "FACT_OBSERVED":
            domain = ev.get("domain")
            subject = ev.get("subject")
            if not domain or not subject:
                continue
            key = f"{domain}:{subject}"
            if key not in store.facts:
                continue
            for f in store.facts[key]:
                if f.value == ev.get("value") and f.valid_from <= t_event <= f.valid_to:
                    f.accesses += 1


def extract_as_of_day(q: Dict) -> Optional[int]:
    for field in ["as_of_day", "day", "t", "time", "timestamp"]:
        val = q.get(field)
        if isinstance(val, (int, float)):
            return int(val)
    return None

In [ ]:
# --------------------------------------------------------------------------------
# STEP 1: DEFINE TASKS
# The @task decorator turns a standard Python function into a Benchmark task.
# The first parameter must always be `llm` (the model being tested).
# --------------------------------------------------------------------------------

@kbench.task(name="AsofQA", description="As of day X, what was true about subject Y?")
def asofqa_task(llm) -> None:
    """AsOfQA: Query facts at a specific point in time."""
    store, event_log = build_store(facts_data, events_data, TemporalAttentionStoreWithValidity)
    
    asof_questions = [q for q in questions if q.get("task_family") == "AsOfQA"]
    
    correct = 0
    total = 0
    
    for q in asof_questions:
        as_of_day = extract_as_of_day(q)
        domain = q.get("domain")
        subject = q.get("subject")
        if as_of_day is None or not domain or not subject:
            continue
        
        key = f"{domain}:{subject}"
        replay_events(store, event_log, as_of_day)
        predicted = store.get(key, as_of_day)
        
        answer = q.get("answer") or ""
        if predicted and answer:
            # Simple check: answer should be in predicted
            kbench.assertions.assert_true(
                answer.lower() in predicted.lower() or predicted.lower() in answer.lower(),
                f"Expected answer containing '{answer}', got '{predicted}'"
            )
            correct += 1
        total += 1
    
    print(f"AsOfQA: {correct}/{total} correct")

In [ ]:
# --------------------------------------------------------------------------------
# STEP 2: RUN ALL TASKS
# We use `kbench.llm` as a placeholder. This allows Kaggle to automatically swap
# in different models later when you use the "Add Models" button in the UI.
# --------------------------------------------------------------------------------

print("Running TemporalBench v1 AsofQA task...")
print("=" * 60)

asofqa_task.run(kbench.llm)

print()
print("=" * 60)
print("TemporalBench AsofQA task completed.")

# --------------------------------------------------------------------------------
# STEP 3: NEXT STEPS
# 1. Click "Save Task" (top right) to publish to the leaderboard.
# 2. Try %autopilot in a new cell to auto-generate tasks or write your own!
# --------------------------------------------------------------------------------